In [ ]:
pip install tensorflow numpy

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Activation
import tensorflow as tf
import random
import sys

#1. Load text
print("Task 1: Loading text...")
# Downloading a sample text file (Shakespeare) for the dataset
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
text = open(path_to_file, 'rb').read().decode(encoding='utf-8').lower()

# To speed up training a slice for the text is used
text = text[:100000]
print(f"Loaded text length: {len(text)} characters.")

Task 1: Loading text...
1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Loaded text length: 100000 characters.


In [ ]:
#2. Create character vocabulary & vectorize
print("Task 2: Creating character vocabulary...")
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Total unique characters (Vocabulary size): {vocab_size}")

# Create dictionaries to map characters to numbers and vice versa
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}

Task 2: Creating character vocabulary...
Total unique characters (Vocabulary size): 37


In [ ]:
import numpy as np
# 3. Creating training sequences
maxlen = 40  # Sequence length to learn from
step = 3     # Step size to slide the window
sentences = []
next_chars = []

# Extract sequences and their corresponding next character
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i : i + maxlen])
    next_chars.append(text[i + maxlen])
print(f"Number of training sequences: {len(sentences)}")

# Vectorization: Convert text sequences to one-hot encoded boolean arrays
X = np.zeros((len(sentences), maxlen, vocab_size), dtype=np.bool_)
y = np.zeros((len(sentences), vocab_size), dtype=np.bool_)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_index[char]] = 1
    y[i, char_to_index[next_chars[i]]] = 1

Number of training sequences: 33320


In [ ]:
#4. Building models
# 1. Simple RNN Model
def build_rnn_model():
    model = Sequential([
        SimpleRNN(128, input_shape=(maxlen, vocab_size)),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

# 2. LSTM Model
def build_lstm_model():
    model = Sequential([
        LSTM(128, input_shape=(maxlen, vocab_size)),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

rnn_model = build_rnn_model()
lstm_model = build_lstm_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#Task 5: Training the models
epochs = 5 # Set higher (e.g., 20-50) for much better text generation
batch_size = 128

print("\n--- Training Simple RNN ---")
rnn_model.fit(X, y, batch_size=batch_size, epochs=epochs)

print("\n--- Training LSTM ---")
lstm_model.fit(X, y, batch_size=batch_size, epochs=epochs)


--- Training Simple RNN ---
Epoch 1/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 12s 39ms/step - loss: 3.0265
Epoch 2/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 19s 35ms/step - loss: 2.4483
Epoch 3/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 2.2690
Epoch 4/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - loss: 2.1558
Epoch 5/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - loss: 2.1097

--- Training LSTM ---
Epoch 1/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 30s 108ms/step - loss: 3.1284
Epoch 2/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 104ms/step - loss: 2.6365
Epoch 3/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 104ms/step - loss: 2.3840
Epoch 4/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 102ms/step - loss: 2.2482
Epoch 5/5
261/261 ━━━━━━━━━━━━━━━━━━━━ 31s 117ms/step - loss: 2.1732


In [ ]:
# 6. Write text generation function
def sample(preds, temperature=1.0):
    """Helper function to sample an index from a probability array."""
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature # Added epsilon to avoid log(0)
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(model, seed_text, num_chars_to_generate=200, temperature=0.5):
    """Generates new text using the trained model."""
    generated = seed_text
    current_seq = seed_text

    for _ in range(num_chars_to_generate):
        # Vectorize current sequence
        x_pred = np.zeros((1, maxlen, vocab_size))
        for t, char in enumerate(current_seq):
            if char in char_to_index:
                x_pred[0, t, char_to_index[char]] = 1

        # Predict next character
        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = index_to_char[next_index]

        # Append to generated text and shift the sequence window
        generated += next_char
        current_seq = current_seq[1:] + next_char

    return generated

In [ ]:
#7. Generating new text

# Pick a random starting sequence from the text
start_index = random.randint(0, len(text) - maxlen - 1)
seed_text = text[start_index : start_index + maxlen]

print(f"\n--- Seed Text ---\n'{seed_text}'\n")

print("--- Generated by Simple RNN ---")
print(generate_text(rnn_model, seed_text, num_chars_to_generate=150, temperature=0.5))

print("\n--- Generated by LSTM ---")
print(generate_text(lstm_model, seed_text, num_chars_to_generate=150, temperature=0.5))


--- Seed Text ---
's 'we thank the gods
our rome hath such '

--- Generated by Simple RNN ---
s 'we thank the gods
our rome hath such ie mungsins at the burs fir mane io dare be ton the thenous the whe sur then wery ur thet you hear iust the gate so thes ands colend the sers ourt.

m

--- Generated by LSTM ---
s 'we thank the gods
our rome hath such the mand at you too dous fore ho the core,
be with the sere tor were the the thin withe the hats and we pond be the th forenius in the corothe the ser
